# Module 2 · Production training, tuning and tracking
    **Outcomes:** (c) organise training into reproducible scripts, (d) tune with Optuna and track runs in MLflow.

> **How to use this notebook.** Run cells top to bottom with **Shift + Enter**.
> Cells marked **✏️ TODO** are yours to complete. **🏁 Checkpoint** cells tell you when a module is done.
> Stuck? Ask your neighbour first, then the trainer. Solutions are released after the session.

In [ ]:
import json, subprocess, sys

import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split

from course_utils import REPO_ROOT

SCRIPTS = REPO_ROOT / "scripts"
print("Training scripts:", sorted(p.name for p in SCRIPTS.iterdir() if p.suffix in (".py", ".yaml")))

## A · From notebook to modular scripts
| File | One responsibility |
|---|---|
| `scripts/config.yaml` | every setting: data, model, paths |
| `scripts/data_loader.py` | load the data, check the target, split |
| `scripts/features.py` | impute and encode |
| `scripts/train.py` | build the model from config and save it |
| `scripts/evaluate.py` | write precision, recall and F1 to JSON |
| `scripts/main.py` | read the config, log, run each step |

In a terminal (repo root, environment active) this is one command:
```bash
python scripts/main.py --config scripts/config.yaml
```
The cell below runs exactly that.

In [ ]:
def run_training():
    out = subprocess.run([sys.executable, str(SCRIPTS / "main.py"), "--config", str(SCRIPTS / "config.yaml")],
                         capture_output=True, text=True)
    print((out.stderr or out.stdout)[-800:])
    if out.returncode:
        raise RuntimeError("Training failed: read the log above")
    return json.loads((REPO_ROOT / "artifacts" / "metrics.json").read_text())

run1 = run_training()
print({k: round(v, 3) for k, v in run1.items() if isinstance(v, float)})

**✏️ Hands-on:** open `scripts/config.yaml` in JupyterLab, change `test_size` from `0.2` to `0.3`
(or `max_iter`), **save**, then run the next cell and compare. Same config + same seed = same result.
Put it back to `0.2` afterwards.

In [ ]:
run2 = run_training()
for k in ["accuracy", "precision", "recall", "f1"]:
    print(f"{k:<10} run 1 {run1[k]:.3f}   run 2 {run2[k]:.3f}")

## B · Hyperparameter tuning with Optuna (20 min)
**Parameters** are learned by `fit()`. **Hyperparameters** are chosen by you, so search for them,
and **never let the test set choose**: score each trial with cross-validation on the training rows.

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

guess = RandomForestClassifier(n_estimators=10, max_depth=2, random_state=42).fit(X_train, y_train)
baseline_auc = roc_auc_score(y_test, guess.predict_proba(X_test)[:, 1])
print(f"Guessed hyperparameters · test ROC-AUC {baseline_auc:.3f}")

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # ✏️ TODO 1: suggest n_estimators (50–300), max_depth (3–20), min_samples_split (2–15)
    #           hint: trial.suggest_int("name", low, high)
    params = {
        "n_estimators": ...,
        "max_depth": ...,
        "min_samples_split": ...,
    }
    clf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    # ✏️ TODO 2: return the mean ROC-AUC of 3-fold cross-validation on X_train, y_train
    #           hint: cross_val_score(clf, X_train, y_train, cv=3, scoring="roc_auc").mean()
    return ...

N_TRIALS = 20   # use 10 on a slow laptop
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS)
print("Best CV ROC-AUC:", round(study.best_value, 4))
print("Best params:", study.best_params)

In [ ]:
trials = study.trials_dataframe()
importance = optuna.importance.get_param_importances(study)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(trials["number"], trials["value"], "o", alpha=0.5, label="trial")
axes[0].plot(trials["number"], trials["value"].cummax(), "-", label="best so far")
axes[0].set(title="Optimisation history", xlabel="trial", ylabel="CV ROC-AUC")
axes[0].legend()
axes[1].barh(list(importance)[::-1], list(importance.values())[::-1])
axes[1].set(title="Which hyperparameter mattered?")
plt.tight_layout()
plt.show()

## C · If it is not logged, it did not happen: MLflow (20 min)
Runs are stored in `mlruns/` at the repo root.

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri((REPO_ROOT / "mlruns").as_uri())
mlflow.set_experiment("day1_loan_default")

with mlflow.start_run(run_name="rf_baseline"):
    mlflow.log_params({"n_estimators": 10, "max_depth": 2})
    mlflow.log_metric("test_auc", baseline_auc)
print("Baseline logged.")

In [ ]:
with mlflow.start_run(run_name="optuna_best_rf"):
    best = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1).fit(X_train, y_train)
    test_auc = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])   # scored ONCE, after tuning

    # ✏️ TODO: log (1) the best params, (2) test_auc and the CV score, (3) a data_version tag, (4) the model
    #   hints: mlflow.log_params(...), mlflow.log_metric("name", value), mlflow.set_tag("name", "value"),
    #          mlflow.sklearn.log_model(best, "model")
    ...

print(f"Logged · test ROC-AUC {test_auc:.4f}")

### 🏁 Checkpoint · Module 2

In [ ]:
runs = mlflow.search_runs(experiment_names=["day1_loan_default"])
cols = [c for c in ["tags.mlflow.runName", "metrics.test_auc", "params.n_estimators", "params.max_depth"] if c in runs]
print(runs[cols].sort_values("metrics.test_auc", ascending=False).head(8).to_string(index=False))

tuned = runs[runs["tags.mlflow.runName"] == "optuna_best_rf"] if "tags.mlflow.runName" in runs else runs.iloc[0:0]
ok = (not tuned.empty and "params.min_samples_split" in tuned and tuned["params.min_samples_split"].notna().any()
      and tuned["metrics.test_auc"].notna().any())
print("\n🏁 Module 2 checkpoint passed. Now open the MLflow UI." if ok
      else "\n❌ No tuned run with params and test_auc yet. Finish the TODO above.")

**Open the MLflow UI.** In a new terminal at the repo root, with the environment active:
```bash
mlflow ui
```
Then browse to http://127.0.0.1:5000, open **day1_loan_default**, sort by `test_auc`, and compare the runs.

### Quick quiz
1. Why does random search often beat grid search?
2. Why must the test set never guide tuning?
3. What should every MLflow run log?